<a href="https://colab.research.google.com/github/ayoub-bakr/RAG-Code/blob/main/MRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# تثبيت المكتبات مع تحديد إصدارات متوافقة لتقليل التعارض
!pip install requests==2.32.4 opentelemetry-api==1.38.0 opentelemetry-sdk==1.38.0
!pip install langchain langchain-community chromadb pypdf tiktoken sentence-transformers transformers accelerate --quiet

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
# تم تغيير المسار من langchain.text_splitter إلى langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

In [ ]:
loader = PyPDFLoader("/content/iso27001.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

In [ ]:
print(chunks[0].page_content)

In [ ]:
# visualization
import matplotlib.pyplot as plt

# تعريف القيم التي تم استخدامها في التقسيم
chunk_size = 300
chunk_overlap = 50

pages = [d.metadata.get("page", 0) for d in chunks]

unique_pages = sorted(set(pages))
counts_per_page = [pages.count(p) for p in unique_pages]

plt.figure(figsize=(10,5))
plt.bar(unique_pages, counts_per_page, color='skyblue')
plt.xlabel("Page number")
plt.ylabel("Number of chunks")
plt.title(f"Chunk distribution per page (chunk_size={chunk_size}, overlap={chunk_overlap})")
plt.show()

In [ ]:
embedings = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
db = Chroma.from_documents(
    documents = chunks,
    embedding = embedings,
    persist_directory = "./chroma_db"
)
retriever = db.as_retriever(search_type = "mmr" ,  search_kwargs={"k": 5})

In [ ]:
pipe = pipeline("text-generation", model="google/flan-t5-base", max_length=200)
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
def ask(question):
    # Use .invoke() instead of the deprecated .get_relevant_documents()
    docs = retriever.invoke(question)
    context = "\n\n".join([f"{d.page_content}\n(Source: page {d.metadata.get('page', '?')})" for d in docs])

    prompt = f"""
Answer the question using ONLY the context below.
Cite the sources (page numbers).

Context:
{context}

Question:
{question}
"""
    return llm.generate([prompt]).generations[0][0].text

In [ ]:
# test question
question = "What is this document about?"
answer = ask(question)
print(answer)